# vesuvius-inkcheck: Colab quickstart (about 5 minutes, CPU only)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dlescanogithub/vesuvius-inkcheck/blob/main/notebooks/quickstart_colab.ipynb)

**What it solves.** An ink map can look convincing and still be noise, texture, or a copy of its own training labels.
`inkcheck` asks one testable question: does the map agree with *known* ink better than the same map placed at random wrong positions?
It returns **PASS / WEAK / FAIL**, a negative control that must FAIL, and a warning when the map is too saturated to judge.

Docs: https://dlescanogithub.github.io/vesuvius-inkcheck/ · PyPI: `vesuvius-inkcheck` · DOI: [10.5281/zenodo.23769415](https://doi.org/10.5281/zenodo.23769415)

**Data.** Nothing from the scrolls ships with this notebook. The example files (about 2 MB) are fetched at run time from public sources
(sha256-checked) into `~/.cache/inkcheck`. Vesuvius Challenge data are CC BY-NC 4.0 with extra terms: **do not commit or redistribute**
the downloaded files or images made from them. The notebook draws no images.

## 1. Install from PyPI

In [ ]:
%pip install -q vesuvius-inkcheck

In [ ]:
import json, os, time, importlib.metadata as im
import numpy as np
from inkcheck import opendata as od
from inkcheck.quick import run as quick
from inkcheck.node_d_v2 import saturation_check

print("vesuvius-inkcheck", im.version("vesuvius-inkcheck"))
WORKERS = max(1, min(4, os.cpu_count() or 1))   # CPU processes for the null tests (Colab free tier: 2)
T0 = time.time()

## 2. Node A + node N on a real segment: `inkcheck quick`

The built-in `343p-ir` example takes the Vesuvius Challenge team's canonical 2.2 µm ink map of PHerc0343P (segment 20250511003658,
on its 8.64 µm canvas) as the "model", and as the reference the letters the team traced on an **infrared photograph** of the fragment,
a reference that comes from neither a model nor the CT.

* **Node A** (does the model see known ink?): band-pass correlation and stroke recall against 192 shifted/rotated copies of the map.
* **Node N** (does the control fail?): the same map moved to the wrong place must FAIL; if it passed, the check would be fooled by texture.

Exit code: 0 = PASS, 1 = WEAK, 2 = FAIL, 3 = INVALID (the control passed). Takes about 2 minutes on 2 cores.

In [ ]:
rc = quick(["343p-ir", "--workers", str(WORKERS), "--out", "out/colab/343p-ir"])
s = json.load(open("out/colab/343p-ir.summary.json"))
print("\nexit code:", rc, {0: "PASS", 1: "WEAK", 2: "FAIL", 3: "INVALID"}.get(rc))
print("protocol verdict:", s["verdict"], "| node A:", s["node_A"], "| wrong-place control:", s["control_verdict"])
assert s["verdict"] == "PASS" and s["control_verdict"] == "FAIL", s

How to read the verdict:

| verdict | meaning | next step |
|---|---|---|
| **PASS** | significant *and* large agreement with known ink, at letter level | stop: "readable" with this model |
| **WEAK** | beats the null, but too small or too few strokes recovered | node B: is the scan to blame? |
| **FAIL** | no better than the same map at a wrong place | node B/C: scan or scroll? |

PASS means "agrees with this reference", not ground truth; with fewer than 10 reference strokes the stroke test has little power.

## 3. The saturation warning: `saturation_check`

A top-quantile agreement test is unreliable on an 8-bit map that is mostly bright: the "top 5%" becomes an arbitrary slice of a plateau.
`saturation_check(map, valid)` reports the share of valid pixels above the high level and flags `saturated=True` past a fixed threshold.
Run it on any map **before** you trust a PASS. Here: one official map of PHerc0172 (scanned at 53 keV, where Reader V3 saturated in our study)
read straight from the open-data bucket as an 8x preview (`-ds8.jpg`; this one is expected to pass as OK, about 15% high), then a synthetic
saturated map so the warning path is visible.

In [ ]:
SCROLL, SEGDIR = "PHerc0172", "20250917143559-w062_20250917143559205_flatboi"
key = next(k for k in od.list_predictions(SCROLL, SEGDIR) if "july_retreat" in k)
a, meta = od.load_prediction(key)
print({k: meta[k] for k in ("scroll", "segment", "scan", "model", "map_pixel_um", "shape")})

def check_saturation(name, m, valid):
    sat = saturation_check(m, valid)
    flag = "WARN: saturated, do not trust a top-q PASS on this map" if sat["saturated"] else "OK"
    print(f"{name:32s} median {sat['median']:6.1f}  share above high level {sat['frac_high']:.2f}  -> {flag}")
    return sat

sat_official = check_saturation("official july_retreat (0172)", a, a > 0)

rng = np.random.default_rng(0)                         # synthetic, no scroll data
synthetic = np.clip(rng.normal(235, 15, (512, 512)), 0, 255).astype(np.uint8)
sat_synth = check_saturation("synthetic saturated map", synthetic, np.ones_like(synthetic, bool))
assert sat_synth["saturated"]

## Summary

In [ ]:
print(f"node A, PHerc0343P team map vs infrared letters : {s['verdict']} (wrong-place control {s['control_verdict']})")
print(f"saturation, official PHerc0172 map               : {'WARN' if sat_official['saturated'] else 'OK'} (share high {sat_official['frac_high']:.2f})")
print(f"saturation, synthetic saturated map              : {'WARN' if sat_synth['saturated'] else 'OK'}")
print(f"total {time.time() - T0:.0f} s")

**Next.**
* Your own map: `!inkcheck quick PRED REF --pred-um .. --ref-um .. --mask MASK` (local paths or http(s)/s3:// URLs).
* A full one-page readability card (nodes A-D): `inkcheck report case.json`; see the [docs](https://dlescanogithub.github.io/vesuvius-inkcheck/).
* The longer tutorial with official open-data maps and surface volumes: [`notebooks/tutorial_5min.ipynb`](https://github.com/dlescanogithub/vesuvius-inkcheck/blob/main/notebooks/tutorial_5min.ipynb).
* Cite: [10.5281/zenodo.23769415](https://doi.org/10.5281/zenodo.23769415).